[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_computing/01_ieee754_floating_point_representation/first_principles.ipynb)

# Topic 01: IEEE 754 Floating-Point Representation

## 1. First-Principles Intuition & Motivation

A computer word has a fixed number of bits — 16, 32, or 64 — yet we want to represent quantities as small as the Planck length ($\approx 1.6 \times 10^{-35}$ m) and as large as the observable universe ($\approx 8.8 \times 10^{26}$ m) in a single number system.

Fixed-point representation (an integer with an implicit decimal point) fails this test: with 64 bits split evenly, we would cover only about 19 orders of magnitude, with *absolute* precision that is wasteful for large numbers and useless for small ones.

The first-principles insight of floating point is borrowed from scientific notation: separate the *scale* of a number (the exponent) from its *significant digits* (the significand). Precision then becomes **relative** — every number carries roughly the same number of significant digits regardless of magnitude.

### The design problem

We seek a finite set $\mathbb{F} \subset \mathbb{R}$ and a rounding map $\mathrm{fl}: \mathbb{R} \to \mathbb{F}$ such that:

1. **Wide dynamic range**: $\mathbb{F}$ spans hundreds of orders of magnitude.
2. **Uniform relative accuracy**: for every $x$ in the normal range, $\frac{\lvert \mathrm{fl}(x) - x \rvert}{\lvert x \rvert} \le u$ for a single small constant $u$.
3. **Deterministic, portable arithmetic**: the result of $x + y$ must be identical on every conforming machine.
4. **Graceful degradation**: overflow, underflow, and invalid operations must produce well-defined representable results ($\pm\infty$, subnormals, NaN) rather than crashes or garbage.

IEEE 754 (1985, revised 2008 and 2019) is the canonical solution, implemented in essentially all modern CPUs and GPUs.

## 2. Rigorous Mathematical Definitions & Theorem Statements

### Definition 2.1 (Binary floating-point system)

A binary floating-point system is a set $\mathbb{F}(p, e_{\min}, e_{\max})$ of numbers of the form

$$
x = (-1)^s \cdot m \cdot 2^{e}
$$

where $s \in \{0, 1\}$ is the sign, the exponent satisfies $e_{\min} \le e \le e_{\max}$, and the significand $m$ is a $(p+1)$-bit binary fixed-point number:

- **Normalized**: $m = (1.b_1 b_2 \dots b_p)_2 \in [1, 2)$, with the leading 1 *implicit* (not stored).
- **Subnormal** (stored exponent field all zeros, fraction nonzero): $m = (0.b_1 b_2 \dots b_p)_2 \in (0, 1)$ with $e = e_{\min}$.

The stored exponent field $E$ uses a *bias*: $e = E - \mathrm{bias}$ with $\mathrm{bias} = 2^{k-1} - 1$ for a $k$-bit exponent field.

### Definition 2.2 (The standard formats)

| Format | Total bits | Exponent bits $k$ | Fraction bits $p$ | $\varepsilon = 2^{-p}$ | Max finite | Min normal |
|---|---|---|---|---|---|---|
| binary64 (double) | 64 | 11 | 52 | $2^{-52} \approx 2.2 \times 10^{-16}$ | $\approx 1.8 \times 10^{308}$ | $2^{-1022}$ |
| binary32 (float) | 32 | 8 | 23 | $2^{-23} \approx 1.2 \times 10^{-7}$ | $\approx 3.4 \times 10^{38}$ | $2^{-126}$ |
| binary16 (fp16) | 16 | 5 | 10 | $2^{-10} \approx 9.8 \times 10^{-4}$ | $65504$ | $2^{-14} \approx 6.1 \times 10^{-5}$ |
| bfloat16 | 16 | 8 | 7 | $2^{-7} \approx 7.8 \times 10^{-3}$ | $\approx 3.4 \times 10^{38}$ | $2^{-126}$ |

Special encodings: exponent field all zeros with $m \ne 0$ gives subnormals; all zeros with $m = 0$ gives $\pm 0$; all ones with $m = 0$ gives $\pm\infty$; all ones with $m \ne 0$ gives NaN.

### Definition 2.3 (Machine epsilon and unit roundoff)

**Machine epsilon** is the gap between $1$ and the next larger representable number:

$$
\varepsilon_{\mathrm{mach}} = 2^{-p}
$$

**Unit roundoff** is the largest possible relative error committed by round-to-nearest:

$$
u = \frac{1}{2}\varepsilon_{\mathrm{mach}} = 2^{-(p+1)}
$$

For binary64: $\varepsilon_{\mathrm{mach}} = 2^{-52}$ and $u = 2^{-53} \approx 1.11 \times 10^{-16}$.

### Definition 2.4 (Unit in the last place)

For $x \ne 0$ with $2^{e} \le \lvert x \rvert \lt 2^{e+1}$ in the normal range, the **ulp** (unit in the last place) is the spacing between consecutive floats near $x$:

$$
\mathrm{ulp}(x) = 2^{e - p} = 2^{\lfloor \log_2 \lvert x \rvert \rfloor - p}
$$

Consequently $\frac{1}{2}\varepsilon_{\mathrm{mach}} \lvert x \rvert \lt \mathrm{ulp}(x) \le \varepsilon_{\mathrm{mach}} \lvert x \rvert$: absolute spacing scales with magnitude, while relative spacing stays confined to a factor-of-2 band.

### Theorem 2.5 (Standard rounding model)

Let $\mathrm{fl}(\cdot)$ denote round-to-nearest (ties to even). For every real $x$ in the normal range of $\mathbb{F}$,

$$
\mathrm{fl}(x) = x(1 + \delta), \qquad \lvert \delta \rvert \le u
$$

Moreover, IEEE 754 requires each basic operation $\circ \in \{+, -, \times, \div\}$ and $\sqrt{\phantom{x}}$ to be **correctly rounded**: the computed result equals the rounding of the exact result,

$$
\mathrm{fl}(x \circ y) = (x \circ y)(1 + \delta), \qquad \lvert \delta \rvert \le u
$$

This single axiom is the foundation of all rounding-error analysis (Higham 2002, Sec. 2.2).

### Theorem 2.6 (Exact integer range)

Every integer $n$ with $\lvert n \rvert \le 2^{p+1}$ is exactly representable in $\mathbb{F}(p, \cdot, \cdot)$. For binary64 this bound is $2^{53} = 9007199254740992$; beyond it, consecutive integers begin to be skipped (the spacing becomes 2, then 4, and so on).

This is why JavaScript (whose `Number` is binary64) defines `MAX_SAFE_INTEGER` as $2^{53} - 1$, and why storing large array indices in floats is dangerous.

### Theorem 2.7 (Sterbenz lemma)

If $x, y \in \mathbb{F}$ satisfy

$$
\frac{y}{2} \le x \le 2y
$$

then $x - y$ is exactly representable: $\mathrm{fl}(x - y) = x - y$ with zero rounding error.

Subtraction of nearby numbers is thus *exact* — the danger of cancellation (Topic 02) is not that the subtraction errs, but that it amplifies previously committed errors.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Derivation 3.1: The relative error bound of round-to-nearest

**Claim**: for normal $x$, $\frac{\lvert \mathrm{fl}(x) - x \rvert}{\lvert x \rvert} \le u = 2^{-(p+1)}$.

**Proof.** Without loss of generality let $x \gt 0$ and let $2^{e} \le x \lt 2^{e+1}$. In this binade, representable numbers are uniformly spaced with gap $2^{e-p}$. Round-to-nearest maps $x$ to the closest grid point, so the absolute error is at most half the gap:

$$
\lvert \mathrm{fl}(x) - x \rvert \le \frac{1}{2} \cdot 2^{e-p} = 2^{e-p-1}
$$

Dividing by $x \ge 2^{e}$:

$$
\frac{\lvert \mathrm{fl}(x) - x \rvert}{\lvert x \rvert} \le \frac{2^{e-p-1}}{2^{e}} = 2^{-(p+1)} = u \qquad \blacksquare
$$

Note the bound is essentially attained just above a power of two, where $x$ is smallest relative to the gap; just below the next power of two the worst relative error is nearly half as large. Rounding error is *not* uniform within a binade.

### Derivation 3.2: Why $0.1$ is not representable, and the exact stored value

**Claim**: $1/10 \notin \mathbb{F}$ for any binary format.

**Step 1.** A binary float is $m \cdot 2^{e}$ with integer $m$ and integer $e$, i.e. a *dyadic rational* (denominator a power of 2). A rational $a/b$ in lowest terms is dyadic iff $b$ has no prime factor other than 2. Since $10 = 2 \cdot 5$ contains the factor 5, $1/10$ is not dyadic. $\blacksquare$

**Step 2.** The binary expansion is periodic:

$$
0.1_{10} = (0.0\overline{0011})_2 = 0.000110011001100\dots_2
$$

**Step 3.** Rounding to 53 significant bits (binary64) gives the exact stored value

$$
\mathrm{fl}(0.1) = \frac{3602879701896397}{2^{55}} = 0.1000000000000000055511151231257827\dots
$$

Hence `0.1 + 0.2 == 0.3` is false: the left side rounds to $0.30000000000000004\dots$, the right side to a *different* float just below $3/10$ — each within one ulp of $3/10$, but not equal to each other.

### Derivation 3.3: How many decimal digits does binary64 carry?

The resolution of a format with a $(p+1)$-bit significand, measured in decimal digits, is

$$
d = \log_{10} 2^{p+1} = (p+1) \log_{10} 2
$$

For binary64: $d = 53 \times 0.30103 \approx 15.95$ digits. Interpretation:

- Any decimal with at most 15 significant digits survives a round trip decimal $\to$ binary64 $\to$ decimal unchanged.
- 17 significant decimal digits are needed to uniquely identify every binary64 value (which is why Python's `repr` prints up to 17 digits).

Analogously: binary32 carries $\approx 7.2$ digits, fp16 $\approx 3.3$, and bfloat16 $\approx 2.4$.

### Derivation 3.4: Spacing of floats and the ulp formula

**Claim**: in the binade $[2^{e}, 2^{e+1})$ the gap between consecutive floats is exactly $2^{e-p}$.

**Proof.** Numbers in this binade have the form $(1.b_1 \dots b_p)_2 \cdot 2^{e}$. Two consecutive significands differ by exactly one unit in the last fractional place, i.e. by $2^{-p}$. Scaling by $2^{e}$ gives the gap

$$
\Delta = 2^{-p} \cdot 2^{e} = 2^{e-p} \qquad \blacksquare
$$

**Consequences.**

- Near $1$ (binary64): gap $= 2^{-52} \approx 2.2 \times 10^{-16}$.
- Near $2^{53}$: gap $= 2$, so `9007199254740992.0 + 1.0` rounds back to $2^{53}$ (ties to even).
- Near the fp16 maximum $65504$: gap $= 32$; the next candidate value $65536$ overflows to $\infty$.

This geometric grid is why absolute tolerances are meaningless without knowing the data's scale, and why `np.isclose` combines `rtol` and `atol`.

### Derivation 3.5: Subnormals close the underflow gap

Without subnormals, the smallest positive float is $2^{e_{\min}}$ and the next representable value below it is $0$: a *relative* gap of 100%. Then $x \ne y$ would not guarantee $\mathrm{fl}(x - y) \ne 0$, silently breaking code like `if x != y: z = 1/(x - y)`.

**Subnormal construction**: when the exponent field is zero, the implicit leading bit becomes 0 and the exponent freezes at $e_{\min}$:

$$
x = (0.b_1 \dots b_p)_2 \cdot 2^{e_{\min}}
$$

The representable positive numbers now descend uniformly from $2^{e_{\min}}$ down to the smallest subnormal

$$
x_{\min} = 2^{e_{\min} - p}
$$

which is $2^{-1074}$ for binary64 and $2^{-24}$ for fp16.

**Theorem (gradual underflow)**: with subnormals, for floats $x, y$, $\mathrm{fl}(x - y) = 0$ iff $x = y$. This restores exact-subtraction reasoning near zero, at the cost of reduced precision (fewer significant bits) inside the subnormal range — precisely where fp16 gradients die in deep learning (Topic 05).

### Derivation 3.6: fp16 vs bfloat16 — the range/precision trade-off, quantified

Both formats spend 16 bits, one on sign. The design choice is how to split the remaining 15 between exponent bits $k$ and fraction bits $p$, with $k + p = 15$.

**Dynamic range** (ratio of max finite to min normal) grows *doubly exponentially* in $k$, roughly like $2^{2^{k}}$:

- fp16 ($k=5$): normal range $[6.1 \times 10^{-5},\; 6.5 \times 10^{4}]$ — about 9 decades.
- bfloat16 ($k=8$): normal range $[1.2 \times 10^{-38},\; 3.4 \times 10^{38}]$ — about 77 decades, identical to fp32.

**Relative precision** $u = 2^{-(p+1)}$:

- fp16: $u = 2^{-11} \approx 4.9 \times 10^{-4}$ ($\approx 3.3$ decimal digits).
- bfloat16: $u = 2^{-8} \approx 3.9 \times 10^{-3}$ ($\approx 2.4$ decimal digits).

**Conclusion**: fp16 is $8\times$ more precise per value; bfloat16 is astronomically wider in range. Since gradient *magnitudes* in deep networks span many decades while stochastic gradient descent tolerates low per-value precision, bfloat16's allocation usually wins for training — it eliminates the overflow/underflow management (loss scaling) that fp16 requires. Topic 05 develops this fully.

## 4. Computational & Algorithmic Insights

### 4.1 Reading bits in NumPy

The bit-level layout is directly inspectable by viewing a float's bytes as an unsigned integer:

- `np.float64(0.1).view(np.uint64)` exposes the sign/exponent/fraction fields.
- `np.finfo(dtype)` reports `eps`, `tiny` (min normal), `smallest_subnormal`, `max`, and `machep` for any float dtype — the authoritative source, instead of memorized constants.
- `np.nextafter(x, y)` returns the neighboring float from $x$ toward $y$; `np.spacing(x)` returns $\mathrm{ulp}(x)$.

A robust equality test is `np.isclose(a, b, rtol, atol)`, which implements the test $\lvert a - b \rvert \le \mathrm{atol} + \mathrm{rtol} \cdot \lvert b \rvert$ — the additive `atol` handles comparisons near zero, where relative error is undefined.

### 4.2 Hardware behavior worth knowing

- **Correct rounding is free**: CPUs and GPUs implement the $(1+\delta)$ guarantee in hardware for $+, -, \times, \div, \sqrt{\phantom{x}}$; transcendental functions ($\exp$, $\sin$) are typically faithful to about 1 ulp but *not* guaranteed correctly rounded.
- **FMA (fused multiply-add)** computes $\mathrm{fl}(ab + c)$ with a *single* rounding, halving the error of the two-step version and enabling error-free transformations. Compilers may fuse or unfuse silently, changing results in the last ulp.
- **Subnormal penalty**: on many CPUs, arithmetic on subnormals traps to microcode and can run 10–100 times slower; performance-critical code sometimes enables flush-to-zero (FTZ), trading gradual underflow away for speed.
- **Non-determinism**: parallel reductions (GPU atomics, multi-threaded BLAS) sum in data-dependent orders; since floating-point addition is not associative, bitwise reproducibility requires fixed reduction trees.

### 4.3 Rounding modes and their uses

IEEE 754 mandates four rounding-direction attributes: round-to-nearest-even (default), toward $0$, toward $+\infty$, toward $-\infty$. The directed modes enable **interval arithmetic** (rigorous enclosures) and a cheap diagnostic: rerunning a computation under different rounding modes and observing large result changes is a classic smoke test for instability.

**Stochastic rounding** — rounding up with probability proportional to the residual — is a non-IEEE mode increasingly used in low-precision ML training because it makes rounding errors unbiased: $\mathbb{E}[\mathrm{fl}(x)] = x$, so errors average out across many updates instead of systematically truncating small weight changes.

## 5. Real-World Physics & AI/ML Applications

### 5.1 The Patriot missile clock drift (1991)

The Patriot system stored time in tenths of seconds and multiplied by a 24-bit truncation of $0.1$. The representation error of $\approx 9.5 \times 10^{-8}$ per unit accumulated over 100 hours of uptime to $\approx 0.34$ s — at Scud velocity, a tracking-gate error of $\approx 600$ m. The failed intercept killed 28 soldiers. Root cause: treating a non-dyadic constant as exact and letting *absolute* error accumulate over a long time horizon.

### 5.2 Ariane 5 flight 501 (1996)

A 64-bit float describing horizontal velocity was converted to a 16-bit *integer*; the value exceeded 32767, the conversion overflowed, the guidance computer emitted a diagnostic bit pattern interpreted as flight data, and the launcher self-destructed 37 s after liftoff. Moral: range analysis is part of numerical design, and float-to-integer boundaries are where it bites hardest.

### 5.3 Mixed-precision deep learning

Modern accelerators (tensor cores) deliver 8–16 times higher throughput in 16-bit formats. The IEEE anatomy explains the engineering around them:

- fp16's maximum of 65504 makes attention logits and loss values overflow-prone — hence pre-softmax scaling and fp32 accumulation inside matmuls.
- fp16's minimum normal $2^{-14}$ sits *above* many small-gradient magnitudes, so gradients underflow to zero — hence **loss scaling** (multiply the loss by $2^{k}$ before backprop, divide gradients by $2^{k}$ after).
- bfloat16 inherits fp32's exponent, eliminating both failure modes at the cost of $u \approx 3.9 \times 10^{-3}$, absorbed by keeping an fp32 master copy of the weights.
- fp8 formats (E4M3, E5M2) push the same trade-off further for training and inference of large language models.

These patterns are derived quantitatively in [Topic 05](../05_numerical_stability_in_deep_learning/README.md).

### 5.4 Reproducibility in scientific computing

Climate models, molecular dynamics, and RL training runs are chaotic: a one-ulp difference in a single sum can macroscopically diverge trajectories within a few thousand steps. IEEE 754 guarantees bit-reproducibility *only* for a fixed operation order — which parallel schedulers do not provide. Production systems therefore either fix reduction orders (deterministic modes in cuDNN/cuBLAS), use compensated or exact summation (Topic 02), or report ensemble statistics instead of single trajectories.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Location |
|---|---|---|
| Rounding model $\mathrm{fl}(x) = x(1+\delta)$ | Higham, *Accuracy and Stability of Numerical Algorithms* (2002) | Sec. 2.2–2.3 |
| Format anatomy, gradual underflow, cancellation | Goldberg, *What Every Computer Scientist Should Know About Floating-Point Arithmetic* (1991) | Secs. 1–3 |
| Normative format and rounding definitions | IEEE 754-2019 standard | Clauses 3–4 |
| Floating point within numerical linear algebra | Trefethen & Bau, *Numerical Linear Algebra* (1997) | Lecture 13 |
| Sterbenz lemma, FMA, correctly rounded functions | Muller et al., *Handbook of Floating-Point Arithmetic* (2018) | Chs. 4–5, 12 |
| History and rationale of IEEE 754 | Kahan, *Status of IEEE 754* lecture notes (1997) | passim |
| Mixed-precision training practice | Micikevicius et al., *Mixed Precision Training* (ICLR 2018) | Secs. 3–4 |
| NumPy dtype and `finfo` machinery | Harris et al., *Array programming with NumPy*, Nature 585 (2020) | Methods |

**Continue to** [Topic 02: Error Propagation and Stability Tricks](../02_error_propagation_and_stability_tricks/README.md), where the $(1+\delta)$ axiom becomes a calculus of accumulated error.